Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 05 · A desk's VaR scenario P&L: the 509 days, the VaR scenario and the ES tail

`var.scenarios` is a scope's P&L vector, computed by the VaR engine when asked and never stored: one P&L per historical scenario (509 on the demo lake), in float64 as the engine sums. The scope's VaR and expected shortfall come with it, in the schema's metadata, **exactly the engine's figures**. This notebook plots the vector and marks the VaR scenario and the ES tail; it computes no risk figure of its own.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## The scope's vector

`SCOPE` is `DESK:RATES` unless `DRISHTI_EXAMPLE_SCOPE` says otherwise (any scope VaR knows: a node of a hierarchy, `BOOK:…`, `DESK:…`).

In [ ]:
import pandas as pd

f = connect()
pnl = f.measure("var.scenarios", SCOPE, date=DATE)
VAR = float(meta(pnl, "drishti.var"))
ES = float(meta(pnl, "drishti.es"))
CCY = meta(pnl, "drishti.currency", "")
print(f"scope {meta(pnl, 'drishti.scope')}, date {meta(pnl, 'drishti.date')}, {pnl.num_rows} scenarios")
print(f"VaR {VAR:,.0f} {CCY}   ES {ES:,.0f} {CCY}   (the engine's figures)")
df = pnl.to_pandas()
df["SCENARIO_DATE"] = pd.to_datetime(df["SCENARIO_DATE"])
df.head()

## Which scenarios the figures point at

The **VaR scenario** is the scenario whose loss is the engine's VaR (the closest, should the engine interpolate). The **ES tail** is the set of worst scenarios whose average loss is the engine's ES. Both are found from the engine's figures; nothing is recomputed. VAR_GUIDE.md gives the engine's exact conventions.

In [ ]:
losses = (-df["PNL"]).sort_values(ascending=False)          # worst first
var_scenario = (df["PNL"] + VAR).abs().idxmin()
running_mean = losses.expanding().mean()
k = int((running_mean - ES).abs().values.argmin()) + 1
tail = losses.index[:k]
print("VaR scenario:", df.loc[var_scenario, "SCENARIO_DATE"].date(), f"P&L {df.loc[var_scenario, 'PNL']:,.0f}")
print(f"ES tail: the worst {k} scenarios, average loss {losses.iloc[:k].mean():,.0f} (the engine's ES {ES:,.0f})")

## The plot

Each bar is one scenario's P&L on its historical date. The dashed line is minus the VaR; the red point is the VaR scenario; the orange bars are the ES tail.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(df["SCENARIO_DATE"], df["PNL"], width=1.5, color="#8aa4c8", label="scenario P&L")
ax.bar(df.loc[tail, "SCENARIO_DATE"], df.loc[tail, "PNL"], width=2.5, color="#e08a2c", label=f"ES tail ({k} scenarios)")
ax.scatter([df.loc[var_scenario, "SCENARIO_DATE"]], [df.loc[var_scenario, "PNL"]], color="#c0392b", zorder=3, s=40,
           label="VaR scenario")
ax.axhline(-VAR, color="#c0392b", linestyle="--", linewidth=1, label=f"-VaR {VAR:,.0f}")
ax.axhline(-ES, color="#e08a2c", linestyle=":", linewidth=1, label=f"-ES {ES:,.0f}")
ax.set_title(f"{SCOPE}: P&L of each historical scenario ({CCY})")
ax.set_ylabel(f"P&L ({CCY})")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## The same vector, sorted

Sorted from the worst loss, the VaR scenario and the ES tail sit at the left edge.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
ordered = df["PNL"].sort_values().reset_index(drop=True)
ax.plot(ordered.index, ordered.values, color="#4a6fa5")
ax.fill_between(range(k), ordered.values[:k], 0, color="#e08a2c", alpha=0.5, label="ES tail")
ax.axhline(-VAR, color="#c0392b", linestyle="--", linewidth=1, label="-VaR")
ax.set_xlabel("scenarios, worst first")
ax.set_ylabel(f"P&L ({CCY})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## One vector per position

`var.parts` with `by='position'` answers one P&L vector per position of the scope, in float32 as the hot file keeps them (the schema's `drishti.pnl.precision` says so). Their sum is the scope's vector within the parity rule; it is never a substitute for the scope's VaR, which is a quantile of the summed vector, not a sum of VaRs.

In [ ]:
import numpy as np

parts = f.measure("var.parts", SCOPE, date=DATE, by="position")
n = len(df)
matrix = parts.column("PNL").combine_chunks().flatten().to_numpy().reshape(parts.num_rows, n)
print(parts.num_rows, "positions, precision", meta(parts, "drishti.pnl.precision"))
summed = matrix.astype("float64").sum(axis=0)
print("largest difference from the scope's vector:", float(np.abs(summed - df["PNL"].to_numpy()).max()))
parts.select(["KEY", "LABEL", "KIND"]).to_pandas().head()

In [ ]:
f.close()